# Extract MIMIC-IV Vital Signs Time Series (72h)

Extracts the following features over the first **72 hours** of each ICU stay:
- Heart Rate
- Systolic Blood Pressure (SBP)
- Diastolic Blood Pressure (DBP)
- Respiratory Rate
- SpO2

Each vital is paired with a **missingness indicator** (null flag).  
A **mortality label** (hospital_expire_flag) row is appended, giving a final shape of **(11, 72)** per patient:
- Channels 0-9: `[vital_0, null_0, vital_1, null_1, ..., vital_4, null_4]`
- Channel 10: mortality label

Data is resampled to **hourly** frequency. Only stays with vitals covering at least 72 hours are kept.

Output tensors are saved as `.pt` files (train / test split, stratified by mortality).

## 1. Database connection & raw vital sign load

In [1]:
import sys
sys.path.append("../")
import helpers.loader as help

load = help.DataBaseLoader(user="ansongeorge", password='', dbname='mimiciv', schema='mimiciv_derived')
load_mimic_hosp = help.DataBaseLoader(user="ansongeorge", password='', dbname='mimiciv', schema='mimiciv_hosp')

vitalsign_df = load['vitalsign']      # full vitalsign derived table
admissions   = load_mimic_hosp['admissions']

## 2. Build stay_id to subject_id mapping

In [2]:
id_mapping = (
    vitalsign_df
    .drop_duplicates(subset="stay_id")[["subject_id", "stay_id"]]
    .set_index("stay_id")
)
print(f"Unique ICU stays in vitalsign table: {len(id_mapping)}")
id_mapping.head()

Unique ICU stays in vitalsign table: 94446


,subject_id
stay_id,
39553978,10000032
37081114,10000690
39765666,10000980
34592300,10001217
37067082,10001217


## 3. Filter stays with >= 70 measurements per vital (pre-filter)

In [3]:
LOAD_GREATER_N = 70   # minimum raw measurement count to consider a stay

# Maps feature name (as it appears in the vitalsign table) to a short alias
VITALS = {
    "heart_rate": "heart_rate",
    "sbp":        "sbp",
    "dbp":        "dbp",
    "resp_rate":  "resp_rate",
    "spo2":       "spo2",
}

vital_dfs = {}
for col, alias in VITALS.items():
    vital_dfs[alias] = load.query(
        f"""
        WITH base AS (
            SELECT subject_id, stay_id, charttime, {col}
            FROM vitalsign
            WHERE {col} IS NOT NULL
        ),
        counts AS (
            SELECT stay_id, COUNT({col}) AS num
            FROM base
            GROUP BY stay_id
        ),
        filtered AS (
            SELECT stay_id FROM counts WHERE num >= {LOAD_GREATER_N}
        )
        SELECT b.subject_id, b.stay_id, b.charttime, b.{col}
        FROM base b
        INNER JOIN filtered f ON b.stay_id = f.stay_id
        """
    )
    print(f"{alias:>12}: {vital_dfs[alias]['stay_id'].nunique()} stays")

  heart_rate: 34595 stays
         sbp: 33322 stays
         dbp: 33315 stays
   resp_rate: 34232 stays
        spo2: 33640 stays


## 4. Intersect stay_ids present in ALL five vitals

In [4]:
id_sets = [set(df["stay_id"].unique()) for df in vital_dfs.values()]
eligible_stay_ids = list(id_sets[0].intersection(*id_sets[1:]))
print(f"Stays with all 5 vitals and >={LOAD_GREATER_N} measurements each: {len(eligible_stay_ids)}")

Stays with all 5 vitals and >=70 measurements each: 31617


## 5. Build mortality label lookup

In [5]:
expired_subject_ids = set(
    admissions.loc[admissions["hospital_expire_flag"] == 1, "subject_id"]
)
print(f"Deceased patients in admissions table: {len(expired_subject_ids)}")

Deceased patients in admissions table: 11800


## 6. Extract & resample to 72-hour hourly time series

For each eligible stay:
1. Resample each vital to **hourly** frequency (first value per hour, no fill).
2. Keep only the **first 72 hours**; skip stays with fewer than 72 hourly slots.
3. Build interleaved `[vital, null_flag]` channels -> shape `(10, 72)`.
4. Append a constant mortality label row -> shape `(11, 72)`.

In [7]:
import numpy as np
from tqdm import tqdm

N_HOURS       = 72
FEATURE_NAMES = list(VITALS.keys())    # ["heart_rate", "sbp", "dbp", "resp_rate", "spo2"]
N_FEATURES    = len(FEATURE_NAMES)    # 5
N_CHANNELS    = N_FEATURES * 2 + 1   # 10 vital+null channels + 1 label = 11

processed_data = []   # list of (11, 72) arrays
skipped = 0

for stay_id in tqdm(eligible_stay_ids, desc="Stays"):
    subject_id = id_mapping.loc[stay_id]["subject_id"]

    channels = []
    valid = True

    for feat in FEATURE_NAMES:
        stay_df = (
            vital_dfs[feat][vital_dfs[feat]["stay_id"] == stay_id][["charttime", feat]]
            .set_index("charttime")
        )

        # Remove duplicate timestamps within the same hour (keep first)
        stay_df = stay_df[~stay_df.index.duplicated(keep="first")]

        # Resample to hourly; no fill -- missingness is preserved as NaN
        hourly = stay_df.resample("h").first()
        values = hourly[feat].to_numpy(dtype=float)

        if len(values) < N_HOURS:
            valid = False
            break

        values    = values[:N_HOURS]
        null_flags = np.isnan(values).astype(float)

        channels.append(values)      # vital values (NaN where missing)
        channels.append(null_flags)  # missingness indicator (1 = missing)

    if not valid:
        skipped += 1
        continue

    # Mortality label: constant row filled with 0 or 1
    label = 1.0 if subject_id in expired_subject_ids else 0.0
    channels.append(np.full(N_HOURS, label, dtype=float))

    # Stack -> (11, 72)
    processed_data.append(np.vstack(channels))

print(f"\nExtracted : {len(processed_data)} stays")
print(f"Skipped   : {skipped} stays (< {N_HOURS} hourly slots in at least one vital)")
print(f"Shape per sample: ({N_CHANNELS}, {N_HOURS})")

Stays:   0%|          | 0/31617 [00:00<?, ?it/s]

Stays: 100%|██████████| 31617/31617 [21:29<00:00, 24.53it/s]


Extracted : 28781 stays
Skipped   : 2836 stays (< 72 hourly slots in at least one vital)
Shape per sample: (11, 72)


## 7. Stack into a single NumPy array & sanity checks

In [8]:
dataset = np.stack(processed_data, axis=0)  # (N, 11, 72)
print(f"Dataset shape : {dataset.shape}")
print(f"\nChannel layout (axis=1):")
for i, feat in enumerate(FEATURE_NAMES):
    print(f"  [{i*2:2d}] {feat}")
    print(f"  [{i*2+1:2d}] {feat}_null")
print(f"  [10] mortality_label")

mortality_rate = dataset[:, -1, 0].mean() * 100
print(f"\nOverall mortality rate : {mortality_rate:.2f}%")
print(f"NaNs present           : {np.isnan(dataset).any()}  "
      f"(expected -- masked by null-flag channels)")

Dataset shape : (28781, 11, 72)

Channel layout (axis=1):
  [ 0] heart_rate
  [ 1] heart_rate_null
  [ 2] sbp
  [ 3] sbp_null
  [ 4] dbp
  [ 5] dbp_null
  [ 6] resp_rate
  [ 7] resp_rate_null
  [ 8] spo2
  [ 9] spo2_null
  [10] mortality_label

Overall mortality rate : 25.18%
NaNs present           : True  (expected -- masked by null-flag channels)


## 8. Train / test split (stratified by mortality, 75/25)

In [9]:
import torch
from sklearn.model_selection import train_test_split
from helpers.utils import seed_everything

seed_everything(2023)

labels = dataset[:, -1, 0]   # (N,) -- mortality label per patient

X_train, X_test = train_test_split(
    dataset,
    test_size=0.25,
    random_state=2023,
    shuffle=True,
    stratify=labels,
)

print(f"Training set : {X_train.shape}  | mortality {100*X_train[:,-1,0].mean():.2f}%")
print(f"Testing  set : {X_test.shape}   | mortality {100*X_test[:,-1,0].mean():.2f}%")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


Seed set to 2023

Training set : (21585, 11, 72)  | mortality 25.17%
Testing  set : (7196, 11, 72)   | mortality 25.18%


## 9. Save tensors

In [10]:
import os
os.makedirs("../data/mimic4-extract", exist_ok=True)

torch.save(
    torch.from_numpy(X_train.astype(np.float64)),
    "../data/mimic4-extract/TRAIN-mimic4_vitals_72h.pt"
)
torch.save(
    torch.from_numpy(X_test.astype(np.float64)),
    "../data/mimic4-extract/TEST-mimic4_vitals_72h.pt"
)

print("Saved:")
print(f"  TRAIN : ../data/mimic4-extract/TRAIN-mimic4_vitals_72h.pt  shape={tuple(X_train.shape)}")
print(f"  TEST  : ../data/mimic4-extract/TEST-mimic4_vitals_72h.pt   shape={tuple(X_test.shape)}")

Saved:
  TRAIN : ../data/mimic4-extract/TRAIN-mimic4_vitals_72h.pt  shape=(21585, 11, 72)
  TEST  : ../data/mimic4-extract/TEST-mimic4_vitals_72h.pt   shape=(7196, 11, 72)
